In [1]:
import sys
sys.path.append("..")
%load_ext autoreload
%autoreload 2

from src.data import load_flows
from src.features import build_host_windows
from src.models import isolation_scores, fit_logistic
from src.evaluate import precision_at_k, recall_at_k, positive_ranks, host_rank

df = load_flows("../data/Wednesday-workingHours.pcap_ISCX.csv")
print(df.shape)
print(df["Label"].value_counts())
print("Time range:", df["Timestamp"].min(), "to", df["Timestamp"].max())

atk = df[df["is_attack"] == 1]
attacker = atk["Source IP"].value_counts().index[0]
print("\nAttack source IPs:")
print(atk["Source IP"].value_counts().head(5))
print("\nAttack destination IPs:")
print(atk["Destination IP"].value_counts().head(5))
print("\nAttack destination ports:")
print(atk["Destination Port"].value_counts().head(5))

(692703, 87)
Label
BENIGN              440031
DoS Hulk            231073
DoS GoldenEye        10293
DoS slowloris         5796
DoS Slowhttptest      5499
Heartbleed              11
Name: count, dtype: int64
Time range: 2017-05-07 01:00:00 to 2017-05-07 12:59:00

Attack source IPs:
Source IP
172.16.0.1    252672
Name: count, dtype: int64

Attack destination IPs:
Destination IP
192.168.10.50    252661
192.168.10.51        11
Name: count, dtype: int64

Attack destination ports:
Destination Port
80     252661
444        11
Name: count, dtype: int64


In [2]:
feat = build_host_windows(df)
print("rows:", len(feat), "positives:", feat["label"].sum(),
      "fraction:", round(feat["label"].mean(), 4))
print(feat[feat["label"] == 1]["attack_frac"].describe().round(2))

for col in ["flows", "n_dst_ports", "n_dst_ips"]:
    print(f"\n{col}: P@5 = {precision_at_k(feat, col, 5)}, "
          f"R@12 = {round(recall_at_k(feat, col, 12), 2)}")
    print("  positive ranks:", positive_ranks(feat, col)[:12])
    print("  attacker host rank:", host_rank(feat, col, attacker))

rows: 25626 positives: 27 fraction: 0.0011
count    27.00
mean      0.86
std       0.28
min       0.02
25%       0.93
50%       0.99
75%       1.00
max       1.00
Name: attack_frac, dtype: float64

flows: P@5 = 1.0, R@12 = 0.26
  positive ranks: [1, 2, 3, 4, 5, 7, 10, 24, 25, 29, 32, 33]
  attacker host rank: (1, 7689)

n_dst_ports: P@5 = 0.0, R@12 = 0.0
  positive ranks: [22, 23, 29, 30, 33, 34, 35, 36, 39, 42, 44, 46]
  attacker host rank: (5, 7689)

n_dst_ips: P@5 = 0.0, R@12 = 0.0
  positive ranks: [1889, 2748, 2749, 2750, 2751, 2752, 2753, 2754, 2755, 2756, 2757, 2758]
  attacker host rank: (191, 7689)


In [3]:
# ranks of every window under two baselines
feat["flows_rank"] = feat["flows"].rank(ascending=False, method="first").astype(int)
feat["ports_rank"] = feat["n_dst_ports"].rank(ascending=False, method="first").astype(int)

# dominant attack type in each positive window
atk_flows = df[df["is_attack"] == 1]
dom = (atk_flows.groupby(["Source IP", "window"])["Label"]
       .agg(lambda s: s.value_counts().index[0])
       .rename("attack_type").reset_index())
feat = feat.merge(dom, on=["Source IP", "window"], how="left")

mixed = feat[(feat["label"] == 1) & (feat["attack_frac"] < 0.5)]
print("positives with attack_frac < 0.5:", len(mixed), "of", feat["label"].sum())

cols = ["window", "attack_type", "flows", "attack_frac",
        "n_dst_ports", "n_dst_ips", "flows_rank", "ports_rank"]
print(feat[feat["label"] == 1].sort_values("flows_rank")[cols].to_string(index=False))

positives with attack_frac < 0.5: 3 of 27
             window      attack_type  flows  attack_frac  n_dst_ports  n_dst_ips  flows_rank  ports_rank
2017-05-07 10:45:00         DoS Hulk  62430     0.999231           49          2           1          58
2017-05-07 10:50:00         DoS Hulk  62166     1.000000            1          1           2       15146
2017-05-07 10:55:00         DoS Hulk  62116     1.000000            1          1           3       15147
2017-05-07 10:40:00         DoS Hulk  34727     0.998272           61          2           4          46
2017-05-07 11:00:00         DoS Hulk   9597     1.000000            1          1           5       15148
2017-05-07 11:10:00    DoS GoldenEye   6237     0.989258           68          2           7          42
2017-05-07 11:15:00    DoS GoldenEye   4204     0.980733           82          2          10          36
2017-05-07 10:15:00 DoS Slowhttptest   1615     0.930031          114          2          24          22
2017-05-07 10

In [4]:
fri = build_host_windows(
    load_flows("../data/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv"))

feat["lr_score"], coefs = fit_logistic(fri, feat)       # trained on Friday only
print("logistic coefficients (Friday):", coefs)
feat["iso_fri"] = isolation_scores(feat, fit_on=fri)    # forest fit on Friday
feat["iso_wed"] = isolation_scores(feat)                # forest fit on Wednesday

for col in ["flows", "lr_score", "iso_fri", "iso_wed"]:
    print(f"\n{col}: P@5 = {precision_at_k(feat, col, 5)}, "
          f"R@12 = {round(recall_at_k(feat, col, 12), 2)}")
    print("  first 12 positive ranks:", positive_ranks(feat, col)[:12])
    print("  attacker host rank:", host_rank(feat, col, attacker))

logistic coefficients (Friday): {'n_dst_ips': np.float64(-3.89), 'n_dst_ports': np.float64(0.9), 'no_reply_frac': np.float64(-1.85), 'mean_bwd_pkts': np.float64(2.01)}

flows: P@5 = 1.0, R@12 = 0.26
  first 12 positive ranks: [1, 2, 3, 4, 5, 7, 10, 24, 25, 29, 32, 33]
  attacker host rank: (1, 7689)

lr_score: P@5 = 1.0, R@12 = 0.19
  first 12 positive ranks: [1, 2, 3, 4, 5, 55, 69, 85, 94, 304, 361, 740]
  attacker host rank: (1, 7689)

iso_fri: P@5 = 1.0, R@12 = 0.33
  first 12 positive ranks: [1, 2, 3, 4, 5, 6, 7, 11, 12, 14, 16, 18]
  attacker host rank: (1, 7689)

iso_wed: P@5 = 1.0, R@12 = 0.33
  first 12 positive ranks: [1, 2, 3, 4, 5, 6, 7, 11, 12, 13, 14, 18]
  attacker host rank: (1, 7689)
